In [1]:
#for running locally:
#pip install torch torchvision scikit-learn pandas pillow matplotlib seaborn#
#pip install timm

import os
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support, classification_report,
    confusion_matrix, roc_auc_score, roc_curve, auc
)
import matplotlib.pyplot as plt
import seaborn as sns
import timm

# ── Configuration ─────────────────────────────────────────────────────────────

for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".pth"):
            print(os.path.join(root, f))
            
BASE_DIR   = Path("/kaggle/input/datasets/duongnguyenquy/mosquitoes-compsci760")
MODELS_DIR_CNN = Path("/kaggle/input/datasets/ykim753/bestcnnpths")
MODELS_DIR_VT  = Path("/kaggle/input/datasets/ykim753/bestvtpths")
OUTPUT_DIR = Path("/kaggle/working/crop_eval_results")

# BASE_DIR   = Path(".")
# MODELS_DIR = BASE_DIR / "final results - top methods from phase1&2 ver"
# OUTPUT_DIR = BASE_DIR / "crop_eval_results"

DATASETS = {
    "rfdetr": BASE_DIR / "cropped_rfdetr/cropped_rfdetr",
    "yolo":   BASE_DIR / "cropped_yolo/cropped_yolo",
}


ORIGINAL_CLASS_NAMES = [
    'aegypti',
    'albopictus',
    'anopheles',
    'culex',
    'culiseta',
    'japonicus-koreicus',
]
NUM_CLASSES = 6   # model fc layer size — must match

BATCH_SIZE = 32
IMG_SIZE   = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Label encoder (fixed to match training) ───────────────────────────────────

# Reconstruct the exact same LabelEncoder used during training
label_encoder = LabelEncoder()
label_encoder.fit(ORIGINAL_CLASS_NAMES)

# ── Dataset ───────────────────────────────────────────────────────────────────

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

class MosquitoImageDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.image_dir   = image_dir
        self.transform   = transform
        self.image_paths = df["img_fName"].astype(str).values
        self.labels      = df["label_id"].values

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_dir / self.image_paths[idx]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.labels[idx]

# ── Model ─────────────────────────────────────────────────────────────────────

def build_resnet50(num_classes):
    m = models.resnet50(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

def build_model_for_pth(pth_name, num_classes):
    name = pth_name.lower()
    if "deit" in name:
        return timm.create_model("deit_base_patch16_224", pretrained=False, num_classes=num_classes)
    elif "vit" in name:
        return timm.create_model("vit_base_patch16_224", pretrained=False, num_classes=num_classes)
    elif "swin" in name:
        return timm.create_model("swin_base_patch4_window7_224", pretrained=False, num_classes=num_classes)
    else:
        # default: resnet50 (covers all your CNN .pth files)
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        return m

def load_pth(pth_path):
    checkpoint = torch.load(pth_path, map_location=device)
    state = checkpoint.get("model_state_dict", checkpoint)
    model = build_model_for_pth(pth_path.stem, NUM_CLASSES)
    model.load_state_dict(state)
    model.to(device)
    model.eval()
    return model
# ── Inference (matches notebook evaluate_loader) ──────────────────────────────

def evaluate_loader(model, loader):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            all_probs.append(probs.cpu().numpy())
            all_preds.append(preds.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    return {
        "preds":  np.concatenate(all_preds),
        "labels": np.concatenate(all_labels),
        "probs":  np.concatenate(all_probs),
    }

# ── Full eval (matches notebook Cell 33/34) ───────────────────────────────────

def full_eval(data, class_names, split_name, model_name, out_dir):
    all_preds  = data["preds"]
    all_labels = data["labels"]
    all_probs  = data["probs"]

    acc     = accuracy_score(all_labels, all_preds)
    bal_acc = balanced_accuracy_score(all_labels, all_preds)

    precision_per_class, recall_per_class, f1_per_class, support = \
        precision_recall_fscore_support(all_labels, all_preds, average=None, zero_division=0)

    p_macro, r_macro, f1_macro, _ = \
        precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)

    p_weighted, r_weighted, f1_weighted, _ = \
        precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)

    try:
        roc_auc = roc_auc_score(all_labels, all_probs, multi_class="ovr", average="macro")
    except Exception:
        roc_auc = np.nan

    # ── Print (same format as notebook) ──────────────────────────────────────
    print(f"\n{'='*30}")
    print(f"{split_name.upper()} Evaluation: {model_name}")
    print(f"{'='*30}")
    print(f"Accuracy:           {acc:.4f}")
    print(f"Balanced Accuracy:  {bal_acc:.4f}")
    print(f"Macro Precision:    {p_macro:.4f}")
    print(f"Macro Recall:       {r_macro:.4f}")
    print(f"Macro F1:           {f1_macro:.4f}")
    print(f"Weighted F1:        {f1_weighted:.4f}")
    print(f"ROC-AUC:            {roc_auc:.4f}")

    per_class_df = pd.DataFrame({
        "class_name": class_names,
        "precision":  precision_per_class,
        "recall":     recall_per_class,
        "f1":         f1_per_class,
        "support":    support,
    })
    print("\nPer-class metrics:")
    print(per_class_df.to_string(index=False))

    print("\nClassification report:")
    print(classification_report(all_labels, all_preds,
                                target_names=class_names, digits=4, zero_division=0))

    # ── Confusion matrix (sns.heatmap, same as notebook) ─────────────────────
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - {split_name} - {model_name}")
    plt.tight_layout()
    plt.savefig(out_dir / f"cm_{split_name}_{model_name}.png", dpi=150)
    plt.close()

    # ── Per-class bar chart ───────────────────────────────────────────────────
    plot_df = per_class_df.melt(
        id_vars=["class_name", "support"],
        value_vars=["precision", "recall", "f1"],
        var_name="metric", value_name="score"
    )
    plt.figure(figsize=(12, 6))
    sns.barplot(data=plot_df, x="class_name", y="score", hue="metric")
    plt.xticks(rotation=45)
    plt.ylim(0, 1)
    plt.title(f"Per-class Metrics - {split_name} - {model_name}")
    plt.tight_layout()
    plt.savefig(out_dir / f"perclass_{split_name}_{model_name}.png", dpi=150)
    plt.close()

    # ── ROC curve per class ───────────────────────────────────────────────────
    n_classes      = len(class_names)
    all_labels_bin = label_binarize(all_labels, classes=range(n_classes))
    plt.figure(figsize=(10, 8))
    for i, cname in enumerate(class_names):
        fpr, tpr, _ = roc_curve(all_labels_bin[:, i], all_probs[:, i])
        roc_auc_i   = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{cname} (AUC = {roc_auc_i:.3f})")
    plt.plot([0, 1], [0, 1], "k--")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve - {split_name} - {model_name}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out_dir / f"roc_{split_name}_{model_name}.png", dpi=150)
    plt.close()

    return {
        "split":              split_name,
        "model":              model_name,
        "accuracy":           acc,
        "balanced_accuracy":  bal_acc,
        "macro_precision":    p_macro,
        "macro_recall":       r_macro,
        "macro_f1":           f1_macro,
        "weighted_precision": p_weighted,
        "weighted_recall":    r_weighted,
        "weighted_f1":        f1_weighted,
        "roc_auc_ovr":        roc_auc,
    }

# ── Main ──────────────────────────────────────────────────────────────────────

def run_dataset(dataset_name, dataset_dir, all_rows):
    print(f"\n{'='*60}")
    print(f"Dataset: {dataset_name.upper()}")
    print(f"{'='*60}")

    ann_path = dataset_dir / "cropped_annotations.csv"
    if not ann_path.exists():
        print(f"  [SKIP] No annotations CSV at {ann_path}")
        return

    df = pd.read_csv(ann_path)

    # One row per image, majority label
    image_df = (
        df.groupby("img_fName")["class_label"]
        .agg(lambda x: x.mode().iloc[0])
        .reset_index()
    )

    # Drop any class not in the original training set (model never saw it)
    unknown = set(image_df["class_label"].unique()) - set(ORIGINAL_CLASS_NAMES)
    if unknown:
        print(f"  [WARN] Dropping unknown classes not in training set: {unknown}")
        image_df = image_df[image_df["class_label"].isin(ORIGINAL_CLASS_NAMES)]

    # Assign label_id using the SAME label encoder as training
    image_df = image_df.copy()
    image_df["label_id"] = label_encoder.transform(image_df["class_label"])

    print(f"  Total images: {len(image_df)}")
    print(f"  Class distribution:")
    print(image_df["class_label"].value_counts().to_string())

    # Classes actually present in this crop dataset (for display in plots)
    present_label_ids = sorted(image_df["label_id"].unique())
    present_class_names = [ORIGINAL_CLASS_NAMES[i] for i in present_label_ids]
    print(f"  Present classes: {present_class_names}")

    loader = DataLoader(
        MosquitoImageDataset(image_df, dataset_dir, eval_transform),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0
    )

    out_dir = OUTPUT_DIR / dataset_name
    out_dir.mkdir(parents=True, exist_ok=True)

    pth_files = sorted(MODELS_DIR_CNN.glob("*.pth")) + sorted(MODELS_DIR_VT.glob("*.pth"))
    if not pth_files:
        print(f"  [ERROR] No .pth files found in {MODELS_DIR_CNN} or {MODELS_DIR_VT}")
        return
    print(f"\n  Found {len(pth_files)} models: {[p.name for p in pth_files]}")

    for pth_path in pth_files:
        model_name = pth_path.stem
        print(f"\n  Loading: {model_name}")
        try:
            model = load_pth(pth_path)
            data  = evaluate_loader(model, loader)

            # Use only the present classes for per-class display
            # but keep all 6 outputs so label IDs are correct
            row = full_eval(data, ORIGINAL_CLASS_NAMES, dataset_name, model_name, out_dir)
            all_rows.append(row)

        except Exception as e:
            print(f"  [ERROR] {model_name}: {e}")
            import traceback; traceback.print_exc()


def main():
    OUTPUT_DIR.mkdir(exist_ok=True)
    all_rows = []

    for name, directory in DATASETS.items():
        run_dataset(name, directory, all_rows)

    if not all_rows:
        print("No results generated.")
        return

    results_df = pd.DataFrame(all_rows)
    out_csv = OUTPUT_DIR / "crop_eval_results.csv"
    results_df.to_csv(out_csv, index=False)
    print(f"\nSaved results: {out_csv}")

    # ── Comparison chart (balanced_accuracy, macro_f1, roc_auc) ──────────────
    for metric in ["balanced_accuracy", "macro_f1", "roc_auc_ovr"]:
        pivot = results_df.pivot_table(index="model", columns="split", values=metric)
        fig, ax = plt.subplots(figsize=(12, 5))
        pivot.plot(kind="bar", ax=ax)
        ax.set_title(f"{metric} — RF-DETR crops vs YOLO crops")
        ax.set_ylabel(metric)
        ax.set_ylim(0, 1)
        ax.legend(title="Dataset")
        plt.xticks(rotation=30, ha="right")
        plt.tight_layout()
        plt.savefig(OUTPUT_DIR / f"comparison_{metric}.png", dpi=120)
        plt.close()

    print("\n── Summary (Accuracy) ──────────────────────────")
    pivot = results_df.pivot_table(index="model", columns="split", values="accuracy")
    print(pivot.round(4).to_string())
    
    print("\n── Summary (Balanced Accuracy) ──────────────────────────")
    pivot = results_df.pivot_table(index="model", columns="split", values="balanced_accuracy")
    print(pivot.round(4).to_string())
    
    print("\n── Summary (Macro F1) ──────────────────────────")
    pivot = results_df.pivot_table(index="model", columns="split", values="macro_f1")
    print(pivot.round(4).to_string())


if __name__ == "__main__":
    main()

/kaggle/input/datasets/ykim753/bestvtpths/deit_weighted_loss_lr0.0003_bs32_ep5_g1.0_wsqrt_inverse_best.pth
/kaggle/input/datasets/ykim753/bestvtpths/vit_stratified_pf_loss_lr0.0003_bs32_ep5_g1.0_wNone_best.pth
/kaggle/input/datasets/ykim753/bestcnnpths/weighted_loss_winverse_lr0.0005.pth
Using device: cpu

Dataset: RFDETR
  Total images: 1562
  Class distribution:
class_label
culex                 700
albopictus            684
culiseta               93
japonicus-koreicus     71
anopheles              10
aegypti                 4
  Present classes: ['aegypti', 'albopictus', 'anopheles', 'culex', 'culiseta', 'japonicus-koreicus']

  Found 3 models: ['weighted_loss_winverse_lr0.0005.pth', 'deit_weighted_loss_lr0.0003_bs32_ep5_g1.0_wsqrt_inverse_best.pth', 'vit_stratified_pf_loss_lr0.0003_bs32_ep5_g1.0_wNone_best.pth']

  Loading: weighted_loss_winverse_lr0.0005

RFDETR Evaluation: weighted_loss_winverse_lr0.0005
Accuracy:           0.9430
Balanced Accuracy:  0.8803
Macro Precision:    0.8